#Pipeline de AED
Vamos assumir que os dados ja estão organizados, tabulados, limpos, padronizados e pre selecionados em um arquivo.

Caso houver problemas com os dados e dificuldade de importação, consultar o material de[ manipulação de dados](https://colab.research.google.com/drive/1D5k41sxFo4Sx3MSp8ZfMlw9e9fLdBQXB?usp=sharing), este material tambem sera util durante a AED, preprocessamento e modelagem.

Colsulte os exemplos de AED:

[carros](https://colab.research.google.com/drive/1nLQ5TEldVr9hMpYfI7JgGA0e61Ug5TCD?usp=sharing)

[titanic](https://colab.research.google.com/drive/188rsi1RB5AGPo5zv-eA6qe9ARc-iBcJC?usp=sharing)

[Pinguins](https://colab.research.google.com/drive/1WNNgYA-qk08rZeYhfEUWtBtRATG_WydP?usp=sharing)




## Carregando pacotes e dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import skew, kurtosis, kruskal
from skimpy import skim

# Carregar o dataset tratado
df = pd.read_csv("dataset_tratado.csv")

# Visualizar as primeiras linhas para garantir o carregamento correto
df.head()

ModuleNotFoundError: No module named 'pandas'

# informações gerais

In [ ]:
# Visualização das primeiras linhas do dataset tratado
df.head()
# Conferir tipos de dados e possíveis problemas
df.info()
# Conferir valores nulos
df.isnull().sum()
# Exemplo de renomeação de colunas para o dataset de carros
df = df.rename(columns={
    'company': 'marca',
    'model': 'modelo',
    'engine': 'motor',
    'cc': 'cilindrada',
    'hp': 'potencia',
    'max_speed': 'velocidade_maxima',
    '0_100_sec': 'tempo_0_100',
    'price': 'preco',
    'fuel_type': 'combustivel',
    'seats': 'assentos',
    'torque': 'torque',
    'battery_capacity': 'capacidade_bateria'
})
# Exemplo de filtragem:
# df = df[df['marca'] == 'FERRARI']
# Exemplo de modificação de variável:
# df['preco'] = df['preco'] / 1000 # Preço em milhares
# Remover coluna (se necessário):
# df = df.drop('capacidade_bateria', axis=1)

In [ ]:
# Função para sumarizar variáveis do dataset de carros
def generate_summary(df):
    summary_data = []
    for column in df.columns:
        na_count = df[column].isnull().sum()
        zero_count = (df[column] == 0).sum()
        summary_data.append([
            column,
            df[column].nunique(),
            na_count,
            na_count / len(df) * 100,
            zero_count,
            zero_count / len(df) * 100,
            df[column].dtype
        ])
    summary_df = pd.DataFrame(summary_data, columns=[
        'variavel', 'Únicos', 'NA', '% de NA', 'Zeros', '% de Zeros', 'Tipo'
    ])
    return summary_df

# Usando a função criada
generate_summary(df)

In [ ]:
# Visualização gráfica dos valores ausentes no dataset de carros
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Visualização de Valores Ausentes (NA)')
plt.xlabel('Colunas')
plt.ylabel('Linhas')
plt.show()

In [ ]:
# Visão geral detalhada do dataset de carros
#!pip install skimpy
from skimpy import skim
skim(df)

# Analise univariada

Apartir de agora vamos analisar cada uma das variaveis individualmente

In [ ]:
# Exemplo de tratamento de variáveis categóricas e numéricas para o dataset de carros
# Transformar 'marca' e 'combustivel' em categorias
df['marca'] = df['marca'].astype('category')
df['combustivel'] = df['combustivel'].astype('category')
# Exemplo de categorizar preço
faixas_preco = [0, 200000, 1000000, float("inf")]
categorias_preco = ["baixo", "medio", "alto"]
df["categoria_preco"] = pd.cut(df["preco"], bins=faixas_preco, labels=categorias_preco)
# Ver valores únicos das categorias
for coluna in df.select_dtypes(include="category"):
    print(f"{coluna}: {df[coluna].unique()}")

Apartir deste ponto vamos analisar categoricos e numericos de maneira distinta

In [ ]:
# Estatísticas descritivas para variáveis numéricas do dataset de carros
from scipy.stats import shapiro, skew, kurtosis
def full_summary(df):
    summary = df.describe().T
    numeric_cols = df.select_dtypes(include=['number']).columns
    summary['IQR'] =  df[numeric_cols].apply(lambda x: x.quantile(0.75) - x.quantile(0.25))
    summary['skewness'] = df[numeric_cols].apply(lambda x: skew(x.dropna()))
    summary['kurtosis'] = df[numeric_cols].apply(lambda x: kurtosis(x.dropna()))
    summary['shapiro_p_value'] = df[numeric_cols].apply(lambda x: shapiro(x.dropna())[1] if len(x.dropna()) > 3 else np.nan)
    return summary
full_summary(df)

In [ ]:
# Histogramas para variáveis numéricas do dataset de carros
for column in df.select_dtypes(include=['number']).columns:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[column], kde=True, bins=20)
    plt.axvline(np.mean(df[column]), ls='--', c='r', label="Média")
    plt.axvline(np.median(df[column]), ls=':', c='g', label="Mediana")
    plt.title(f'Histograma de {column}')
    plt.xlabel(column)
    plt.ylabel('Frequência')
    plt.legend()
    plt.show()

In [ ]:
# QQ-plot para variáveis numéricas do dataset de carros
for column in df.select_dtypes(include=['number']).columns:
    plt.figure(figsize=(8, 6))
    sm.qqplot(df[column].dropna(), line='s')
    plt.title(f'QQ-plot de {column}')
    plt.xlabel(column)
    plt.ylabel('Frequência')
    plt.show()

In [ ]:
# Boxplot e violino para variáveis numéricas do dataset de carros
for column in df.select_dtypes(include=['number']).columns:
    plt.figure(figsize=(5, 8))
    sns.boxplot(y=df[column])
    plt.title(f'Boxplot de {column}')
    plt.xlabel(column)
    plt.show()

for column in df.select_dtypes(include=['number']).columns:
    plt.figure(figsize=(5, 8))
    sns.violinplot(y=df[column])
    plt.title(f'Violinplot de {column}')
    plt.xlabel(column)
    plt.show()

Agora as colunas categoricas

In [ ]:
# Percentual de cada categoria para 'marca'
cat_porcentagem = df['marca'].value_counts(normalize=True) * 100
cat_porcentagem.plot(kind='bar', color='skyblue')
for i, valor in enumerate(cat_porcentagem):
    plt.text(i, valor, f'{valor:.2f}%', ha='center', va='bottom')
plt.ylabel('Percentual (%)')
plt.title('Distribuição Percentual das Marcas')
plt.show()

In [ ]:
# Número absoluto e percentual de cada tipo de combustível
counts = df['combustivel'].value_counts()
percentages = df['combustivel'].value_counts(normalize=True) * 100
pd.DataFrame({'Counts': counts, 'Percentages': percentages})

## Analise Multivariada

Agora vamos buscar a relação entre as variáveis. Em especial com a nossa variável alvo.

#### Categorico(alvo) x Numerico

In [ ]:
# Exemplo: distribuição de preço por tipo de combustível
sns.kdeplot(data=df, x="preco", hue='combustivel', fill=True, common_norm=False, clip=(df['preco'].min(), df['preco'].max()))
plt.title('Distribuição de Preço por Tipo de Combustível')
plt.show()

In [ ]:
# Boxplot de preço por tipo de combustível
plt.figure(figsize=(10, 6))
sns.boxplot(x='combustivel', y='preco', data=df)
plt.title('Boxplot de Preço por Tipo de Combustível')
plt.xlabel('Tipo de Combustível')
plt.ylabel('Preço')
plt.show()

#### categorico x categorico



In [ ]:
# Tabela de contingência entre marca e tipo de combustível
cross_tab = pd.crosstab(df['marca'], df['combustivel'], normalize='index')
print(cross_tab)
ax = cross_tab.plot(kind='bar', stacked=True, width=0.8, figsize=(12, 5))
for p in ax.patches:
    width = p.get_width()
    height = p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width / 2, y + height / 2, f'{height:.1%}', ha='center', va='center', color='white', fontsize=10)
plt.title('Distribuição de Combustível por Marca')
plt.show()

####Numerico x Numerico

In [ ]:
# Gráfico de dispersão entre preço e potência
sns.lmplot(x='potencia', y='preco', data=df, ci=None)
plt.title('Dispersão entre Potência e Preço')
plt.show()

#### Multivariada geral e correlações


In [ ]:
# Pairplot para análise multivariada
sns.pairplot(df.dropna(), hue="combustivel")
plt.show()
sns.pairplot(df, kind='reg', diag_kind='hist')
plt.show()

In [ ]:
#correlações
#!pip install dython

from dython import nominal

correlation_matrix = nominal.associations(df, figsize=(10,10), mark_columns=True, num_num_assoc='spearman')

# Mostra a matriz de correlação
print(correlation_matrix)

In [ ]:
# Seleção de features para o dataset de carros usando preço como alvo
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.impute import SimpleImputer

# 1. Separar features (X) e alvo (y)
X = df.drop(['preco'], axis=1).select_dtypes(include=['number'])
y = df['preco']

# 2. Remover colunas sem nenhum valor observado
X = X.dropna(axis=1, how='all')

# 3. Imputar valores ausentes nas features numéricas
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# 4. Instanciar e configurar o seletor
k_best_selector = SelectKBest(score_func=mutual_info_regression, k=3)

# 5. Treinar o seletor nos dados completos
k_best_selector.fit(X_imputed, y)

# 6. Obter os resultados
all_feature_scores = pd.Series(k_best_selector.scores_, index=X.columns).sort_values(ascending=False)

selected_features_mask = k_best_selector.get_support()
selected_features_names = X.columns[selected_features_mask]

# 7. Exibir os resultados da análise
print("--- Análise de Seleção de Features com SelectKBest ---")
print("\nScores de Informação Mútua para todas as features:")
print(all_feature_scores)
print("\n---------------------------------------------------------")
print(f"As {len(selected_features_names)} melhores features selecionadas (k=3) foram:")
print(list(selected_features_names))

In [ ]:
# AED automatizado
#!pip install ydata-profiling
from ydata_profiling import ProfileReport


# gerar relatorio
report=ProfileReport(df)
report.to_notebook_iframe()